# OmniVoice — Lồng tiếng tiếng Việt trên Colab

Notebook chạy [OmniVoice](https://huggingface.co/k2-fsa/OmniVoice) với pipeline **SRT lồng tiếng** (`native`).

**Luồng Drive:**
1. Cấu hình giọng + tên file SRT ở **mục 0**
2. Đặt file `.srt` vào **`MyDrive/Omivoice/`** (tên = `SRT_FILE` + `.srt`, vd. `input` → `input.srt`)
3. Chạy notebook → kết quả ghi lại **`MyDrive/Omivoice/`**

**Yêu cầu:** Runtime **GPU L4** (Runtime → Change runtime type → L4).

## 0. Cấu hình

In [ ]:
# Hugging Face (giọng + speak.py)
HF_REPO = "STBack23/omnivoice-vi"

# Google Drive — MyDrive/Omivoice/
DRIVE_FOLDER = "Omivoice"
SRT_FILE = "input"   # tên file (không cần .srt) -> input.srt

# Giọng: ban_mai | lan_trinh | ngan_ha | ngoc_huyen | thao_trinh | tuong_vy
DEFAULT_VOICE = "tuong_vy"
MERGE_MODE = "native"  # native | cascade | fit | strict
NUM_STEP = 32

## 1. Kiểm tra GPU & cài đặt

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "Bật GPU L4: Runtime → Change runtime type → Hardware accelerator → L4"
)
gpu = torch.cuda.get_device_name(0)
print(f"GPU: {gpu}")
if "L4" not in gpu.upper():
    print(
        f"⚠ Đang dùng {gpu}, không phải L4 — đổi runtime sang L4 để nhanh hơn."
    )

In [ ]:
# Tắt XET — thường gây treo download ~89% trên Colab.
# PHẢI chạy cell này TRƯỚC mọi `from huggingface_hub import ...`
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"

print("HF: HF_HUB_DISABLE_XET=1, timeout 300s")

In [ ]:
!pip install -q omnivoice audiotsm huggingface_hub soundfile librosa

### Repo private — đăng nhập Hugging Face (một lần)

1. Tạo token **Read** tại [hf.co/settings/tokens](https://huggingface.co/settings/tokens)
2. Colab → **🔑 Khóa bí mật** (Manage secrets) → thêm tên `HF_TOKEN`, dán token → **Lưu**
3. Chỉ cần làm **một lần** — các lần chạy sau notebook tự lấy token, không cần nhập lại

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    token = userdata.get("HF_TOKEN")
except userdata.SecretNotFoundError as e:
    raise RuntimeError(
        "Chưa có khóa bí mật HF_TOKEN.\n"
        "Colab → 🔑 Khóa bí mật → Thêm secret tên HF_TOKEN "
        "(token Read từ hf.co/settings/tokens).\n"
        "Lưu một lần, lần sau không cần nhập lại."
    ) from e

if not token or not str(token).strip():
    raise RuntimeError("HF_TOKEN trong Khóa bí mật đang trống — hãy cập nhật lại.")

login(token.strip())
print("Đã login Hugging Face (HF_TOKEN từ Khóa bí mật)")

## 2. Tải giọng & công cụ từ Hugging Face

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

ROOT = Path(snapshot_download(HF_REPO, repo_type="dataset", local_dir="/content/omnivoice-vi"))
print(f"Đã tải: {ROOT}")
print("Giọng:", [p.parent.name for p in sorted((ROOT / "voices").glob("*/profile.json"))])

In [ ]:
import importlib.util
import sys

speak_path = ROOT / "tools" / "speak.py"
spec = importlib.util.spec_from_file_location("speak", speak_path)
speak = importlib.util.module_from_spec(spec)
sys.modules["speak"] = speak
spec.loader.exec_module(speak)
print("Đã load speak.py")

## 3. Tải & nạp model (~3–8 phút lần đầu)

Lần đầu tải **~3.3 GB**. Treo ở ~89% → **Runtime → Restart session**, chạy lại từ **mục 1** (cell tắt XET phải chạy trước login HF).

In [ ]:
from huggingface_hub import hf_hub_download

OMNI_MODEL = "k2-fsa/OmniVoice"

# Tải từng file lớn (tránh treo snapshot_download trên Colab)
for relpath, label in [
    ("model.safetensors", "LLM ~2.4 GB"),
    ("audio_tokenizer/model.safetensors", "Audio tokenizer ~770 MB"),
]:
    print(f"\n>>> {label}")
    path = hf_hub_download(OMNI_MODEL, relpath)
    print(f"    OK: {path}")

for relpath in (
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "chat_template.jinja",
    "audio_tokenizer/config.json",
    "audio_tokenizer/preprocessor_config.json",
):
    hf_hub_download(OMNI_MODEL, relpath)

print("\n✓ Model k2-fsa/OmniVoice đã cache xong.")

In [ ]:
PROFILE = speak.load_profile(ROOT / "voices" / DEFAULT_VOICE / "profile.json")
MODEL = speak.load_model(PROFILE)
PROMPT = speak.ensure_voice_prompt(MODEL, PROFILE)
print(f"Giọng: {PROFILE['name']} ({DEFAULT_VOICE}) | SR: {MODEL.sampling_rate}")

## 4. Lồng tiếng SRT từ Google Drive

Đặt file SRT vào **`MyDrive/Omivoice/`**. Giọng = **`DEFAULT_VOICE`** ở mục 0.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_OMIVOICE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
DRIVE_OMIVOICE.mkdir(parents=True, exist_ok=True)

srt_name = SRT_FILE if SRT_FILE.lower().endswith(".srt") else f"{SRT_FILE}.srt"
SRT_PATH = DRIVE_OMIVOICE / srt_name

if not SRT_PATH.exists():
    existing = sorted(DRIVE_OMIVOICE.glob("*.srt"))
    hint = "\n  ".join(str(p.name) for p in existing) if existing else "(chưa có file .srt)"
    raise FileNotFoundError(
        f"Không thấy {SRT_PATH}\n"
        f"Đặt file vào MyDrive/{DRIVE_FOLDER}/ hoặc sửa SRT_FILE.\n"
        f"Các file .srt hiện có:\n  {hint}"
    )

MERGE_OUT = DRIVE_OMIVOICE / f"{SRT_PATH.stem}_merged.wav"
OUTPUT_DIR = DRIVE_OMIVOICE / "output" / DEFAULT_VOICE / SRT_PATH.stem

print(f"Drive:  {DRIVE_OMIVOICE}")
print(f"Giọng:  {DEFAULT_VOICE}")
print(f"Input:  {SRT_PATH}")
print(f"Output: {MERGE_OUT}")
print(f"Cues:   {OUTPUT_DIR}")

In [ ]:
from IPython.display import Audio, display
from tqdm.auto import tqdm
import sys

profile_path = ROOT / "voices" / DEFAULT_VOICE / "profile.json"

pbar = tqdm(total=100, unit="%", desc="Chuẩn bị")


def on_log(line: str) -> None:
    tqdm.write(line)
    sys.stdout.flush()

def on_progress(pct: float, desc: str = "") -> None:
    pbar.n = min(100, max(0, int(round(pct * 100))))
    if desc:
        pbar.set_description(desc)
    pbar.refresh()

print(f"=== SRT: {SRT_PATH.name} | giọng {DEFAULT_VOICE} | mode {MERGE_MODE} ===", flush=True)

result = speak.run_srt_pipeline(
    profile_path,
    SRT_PATH,
    output_dir=OUTPUT_DIR,
    merge=True,
    merge_output=MERGE_OUT,
    merge_mode=MERGE_MODE,
    num_step=NUM_STEP,
    model=MODEL,
    prompt=PROMPT,
    on_log=on_log,
    progress=on_progress,
)

pbar.close()

merged = result.get("merged_wav")
if merged:
    display(Audio(str(merged)))

print("\n--- Đã lưu trên Drive ---")
print(f"WAV gộp:     {result.get('merged_wav')}")
print(f"Từng cue:    {result.get('output_dir')}")
print(f"SRT shifted: {result.get('shifted_srt')}")
stats = result.get("stats", {})
print(
    f"Thống kê: {stats.get('generated', 0)} cue mới, "
    f"{stats.get('skipped', 0)} bỏ qua, "
    f"{stats.get('overflow', 0)} tràn"
)

## 5. Giải phóng VRAM & tắt runtime

Giải phóng bộ nhớ GPU rồi **ngắt session Colab** (không tính phí GPU nữa). Chạy cell này sau khi mục 4 xong.

In [ ]:
import gc
import torch
from google.colab import runtime

try:
    del MODEL, PROMPT, result, pbar
except NameError:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Đã giải phóng VRAM")
print("Đang tắt runtime Colab...")
runtime.unassign()